In [0]:
--1. Which stores are genuinely profitable once real discounting is accounted for?

SELECT
store_id ,
ROUND(SUM(profit)/SUM(total_amount)*100,3) AS margin
FROM
fact_sales
GROUP BY
store_id
ORDER BY
margin DESC
LIMIT 5   ;


--2. Which categories drive revenue and margin vs. just heavy discounting?

SELECT
p.category_name,
SUM(f.unit_price * f.quantity)        AS gross_revenue,
SUM(f.total_amount)                   AS net_revenue,
SUM(f.cost_amount)                    AS total_cost,
SUM(f.profit)                         AS profit,
AVG(f.discount_pct)                   AS avg_discount_pct
FROM workspace.meridianhealth.fact_sales f
JOIN workspace.meridianhealth.dim_product p
ON f.product_id = p.product_id
GROUP BY p.category_name ;


--3. How much inventory is at risk of expiring unsold, and where?

SELECT
f.store_id,
p.product_id,
p.product_name,
p.expiry_date,
SUM(f.quantity) AS total_quantity
FROM
workspace.meridianhealth.fact_sales f
JOIN workspace.meridianhealth.dim_product p ON f.product_id = p.product_id
WHERE
p.expiry_date < CURRENT_DATE() OR
p.expiry_date IS NULL OR
p.expiry_date = CURRENT_DATE() + INTERVAL 30 DAYS
GROUP BY f.store_id, p.product_id, p.product_name, p.expiry_date
ORDER BY store_id ASC ;


--4. Is pharmacist staffing matched to prescription volume?

SELECT 
    fp.store_id,
    COUNT(DISTINCT de.employee_id) AS num_pharmacists,
    COUNT(fp.prescription_id) AS total_prescriptions,
    ROUND(COUNT(fp.prescription_id) / COUNT(DISTINCT de.employee_id), 2) AS prescriptions_per_pharmacist
FROM workspace.meridianhealth.fact_prescriptions fp
JOIN workspace.meridianhealth.dim_employee de 
    ON fp.store_id = de.store_id
WHERE is_pharmacist = 1    
GROUP BY fp.store_id
ORDER BY total_prescriptions DESC;


--5. Which customer segments and payment channels matter most?

SELECT 
    dc.loyalty_tier AS customer_segment,
    fs.payment_method,
    ROUND(SUM(fs.total_amount),2) AS total_revenue,
    ROUND(SUM(fs.profit) ,2)AS total_profit,
    COUNT(*) AS num_transactions
FROM workspace.meridianhealth.fact_sales fs
JOIN workspace.meridianhealth.dim_customer dc 
    ON fs.customer_id = dc.customer_id
GROUP BY dc.loyalty_tier, fs.payment_method
ORDER BY total_profit DESC;
